# Random Forest

## 1. Import Libraries

In [9]:
import pandas as pd
import numpy as np
import os
import pickle

from sklearn.ensemble import StackingClassifier, RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score, GridSearchCV
from sklearn.metrics import (accuracy_score, precision_score, recall_score, 
                             f1_score, roc_auc_score)

import warnings
warnings.filterwarnings('ignore')

## Load Data

In [10]:
# 1. LOAD DỮ LIỆU ĐÃ SCALED
data_dir = '../data/'
target_col = 'class'

train_df = pd.read_csv(os.path.join(data_dir, 'train_data.csv'))
test_df = pd.read_csv(os.path.join(data_dir, 'test_data.csv'))

X_train = train_df.drop(columns=[target_col])
y_train = train_df[target_col]
X_test = test_df.drop(columns=[target_col])
y_test = test_df[target_col]

# 2. HÀM ĐÁNH GIÁ (Lưu kết quả chung vào list)
results_list = []

def evaluate_model(model, version_name, X_test, y_test):
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]
    
    res = {
        'Version': version_name,
        'Accuracy': round(accuracy_score(y_test, y_pred), 4),
        'Precision': round(precision_score(y_test, y_pred), 4),
        'Recall': round(recall_score(y_test, y_pred), 4),
        'F1-Score': round(f1_score(y_test, y_pred), 4),
        'AUC': round(roc_auc_score(y_test, y_prob), 4)
    }
    
    results_list.append(res)
    print(f"[{version_name}] Acc: {res['Accuracy']} | F1: {res['F1-Score']} | AUC: {res['AUC']}")
    return model

###  Danh sách lưu kết quả để xuất CSV

In [11]:
results_list = []

def evaluate_model(model, name, X_test, y_test):
    """Dự đoán và trích xuất các chỉ số phân loại theo yêu cầu (Acc, Prec, Rec, F1, AUC)"""
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1] # Lấy xác suất của class 1 để tính AUC
    
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_prob)
    
    res = {
        'Algorithm': name,
        'Accuracy': round(acc, 2),
        'Precision': round(prec, 2),
        'Recall': round(rec, 2),
        'F1-Score': round(f1, 2),
        'AUC': round(auc, 2)
    }
    
    results_list.append(res)
    print(f"[{name}] Acc: {acc:.2f} | Prec: {prec:.2f} | Rec: {rec:.2f} | F1: {f1:.2f} | AUC: {auc:.2f}")
    return res


### 3. CHẠY BASELINE MODEL (Tham số mặc định)

In [12]:
print("--- V1: Baseline Random Forest ---")
# Sử dụng cấu hình mặc định để làm mốc so sánh (Baseline)
RandomForest_baseline_model = RandomForestClassifier(random_state=42)
RandomForest_baseline_model.fit(X_train, y_train)

evaluate_model(RandomForest_baseline_model, "V1: RandomForest Baseline", X_test, y_test)

--- V1: Baseline Random Forest ---
[V1: RandomForest Baseline] Acc: 1.00 | Prec: 1.00 | Rec: 1.00 | F1: 1.00 | AUC: 1.00


{'Algorithm': 'V1: RandomForest Baseline',
 'Accuracy': 1.0,
 'Precision': 1.0,
 'Recall': 1.0,
 'F1-Score': 1.0,
 'AUC': 1.0}

### 4. CHẠY TUNING MODEL (Dùng GridSearchCV)

In [13]:
print("--- V2: GridSearchCV Tuning ---")
param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [None, 10, 20, 30],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['sqrt', 'log2']
}

grid_search = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1
)

grid_search.fit(X_train, y_train)
RandomForest_tuned_model = grid_search.best_estimator_

print(f"Best Params found by CV: {grid_search.best_params_}")
evaluate_model(RandomForest_tuned_model, "V2: RandomForest Tuned (GridSearch)", X_test, y_test)


--- V2: GridSearchCV Tuning ---


Best Params found by CV: {'max_depth': None, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 200}
[V2: RandomForest Tuned (GridSearch)] Acc: 1.00 | Prec: 1.00 | Rec: 1.00 | F1: 1.00 | AUC: 1.00


{'Algorithm': 'V2: RandomForest Tuned (GridSearch)',
 'Accuracy': 1.0,
 'Precision': 1.0,
 'Recall': 1.0,
 'F1-Score': 1.0,
 'AUC': 1.0}

In [14]:
print("--- V3: Ensemble Learning (Stacking) ---")
# Kết hợp Random Forest và KNN làm Base Models
base_estimators = [
    ('rf', RandomForestClassifier(n_estimators=50, random_state=42)),
    ('knn', KNeighborsClassifier(n_neighbors=5))
]

RandomForest_stacking_model = StackingClassifier(
    estimators=base_estimators,
    final_estimator=RandomForestClassifier(n_estimators=100, random_state=42),
    cv=5
)

RandomForest_stacking_model.fit(X_train, y_train)
evaluate_model(RandomForest_stacking_model, "V3: RandomForest Stacking Ensemble", X_test, y_test)

--- V3: Ensemble Learning (Stacking) ---
[V3: RandomForest Stacking Ensemble] Acc: 1.00 | Prec: 0.99 | Rec: 1.00 | F1: 1.00 | AUC: 1.00


{'Algorithm': 'V3: RandomForest Stacking Ensemble',
 'Accuracy': 1.0,
 'Precision': 0.99,
 'Recall': 1.0,
 'F1-Score': 1.0,
 'AUC': 1.0}

### (5) Lưu kết quả

In [15]:
# CẤU HÌNH ĐƯỜNG DẪN LƯU
save_path = '../experiment/Random Forest/'
os.makedirs(save_path, exist_ok=True)

# 1. TỔNG HỢP & LƯU CSV
df_results = pd.DataFrame(results_list)
csv_output = os.path.join(save_path, 'Random_Forest_evaluation_results.csv')
df_results.to_csv(csv_output, index=False)

# 2. LƯU MODELS (.pkl)
with open(os.path.join(save_path, 'Random_Forest_baseline.pkl'), 'wb') as f:
    pickle.dump(RandomForest_baseline_model, f)

with open(os.path.join(save_path, 'Random_Forest_tuned.pkl'), 'wb') as f:
    pickle.dump(RandomForest_tuned_model, f)

with open(os.path.join(save_path, 'Random_Forest_stacking.pkl'), 'wb') as f:
    pickle.dump(RandomForest_stacking_model, f)

print("-" * 40)
print(f"✅ Đã lưu file CSV tại: {csv_output}")
print(f"✅ Đã lưu models tại: {save_path}")
print("-" * 40)

display(df_results)

----------------------------------------
✅ Đã lưu file CSV tại: ../experiment/Random Forest/Random_Forest_evaluation_results.csv
✅ Đã lưu models tại: ../experiment/Random Forest/
----------------------------------------


,Algorithm,Accuracy,Precision,Recall,F1-Score,AUC
0,V1: RandomForest Baseline,1.0,1.00,1.0,1.0,1.0
1,V2: RandomForest Tuned (GridSearch),1.0,1.00,1.0,1.0,1.0
2,V3: RandomForest Stacking Ensemble,1.0,0.99,1.0,1.0,1.0


In [16]:
!jupyter nbconvert --to html Random_Forest_model.ipynb

[NbConvertApp] Converting notebook Random_Forest_model.ipynb to html
[NbConvertApp] Writing 306635 bytes to Random_Forest_model.html
